In [1]:
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
import pandas as pd
import matplotlib.pyplot as plt
import ast
import sacrebleu

In [2]:
# nltk.download('punkt')

In [3]:
def calculate_sacrebleu(predicted, references):
    # `references` must be a list of reference translations for each source sentence.
    # Each reference should be in a list even if there is only one reference per source.
    references = [[ref] for ref in references]
    
    # `predicted` must be a list where each element is a single system's output
    predicted = [predicted]
    
    # Calculate BLEU score
    bleu_score = sacrebleu.corpus_bleu(predicted, references)
    return {
        "score": bleu_score.score,  # The BLEU score
        "precisions": bleu_score.precisions,  # Precisions for each n-gram length
        "bp": bleu_score.bp,  # Brevity penalty
        "sys_len": bleu_score.sys_len,  # System length
        "ref_len": bleu_score.ref_len  # Reference length
    }

In [4]:
def remove_final_period(tokens):
    # Remove the period at the end of the list if it exists
    if tokens and tokens[-1] == '.':
        return tokens[:-1]
    return tokens

import re
def tokenizer_indonesian(text):
    text = text.lower()
    text = re.sub(r"[^\w\s-]", "", text)
    return text.split()


def calculate_bleu(predicted, references):
    # Tokenizing the sentences
    reference_tokens = [tokenizer_indonesian(ref) for ref in references]
    predicted_tokens = tokenizer_indonesian(predicted)

    # Use a smoothing function
    chencherry = SmoothingFunction()
    smooth = chencherry.method1

    # Calculating BLEU scores for different n-gram lengths
    bleu_1 = sentence_bleu(reference_tokens, predicted_tokens, weights=(1, 0, 0, 0), smoothing_function=smooth)
    bleu_2 = sentence_bleu(reference_tokens, predicted_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
    bleu_3 = sentence_bleu(reference_tokens, predicted_tokens, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
    bleu_4 = sentence_bleu(reference_tokens, predicted_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)

    return {"bleu_1": bleu_1, "bleu_2":bleu_2, "bleu_3":bleu_3, "bleu_4":bleu_4}

def evaluate(data_root_path, filename):
    df = pd.read_csv(filename, delimiter=',')
    df['target'] = df['target'].apply(ast.literal_eval)
    
    averages = {
        'bleu_1': 0, 
        'bleu_2': 0, 
        'bleu_3': 0, 
        'bleu_4': 0,
    }
    for filename, predicted, target in zip(df['filename'], df['predicted'], df['target']):
        predicted = predicted
        score = calculate_bleu(predicted, target)
        averages['bleu_1']+=score['bleu_1']
        averages['bleu_2']+=score['bleu_2']
        averages['bleu_3']+=score['bleu_3']
        averages['bleu_4']+=score['bleu_4']
    
    n = len(df)
    averages['bleu_1'] /= n
    averages['bleu_2'] /= n
    averages['bleu_3'] /= n
    averages['bleu_4'] /= n

    return averages
 
# result = 'testing_1_100_m.csv'
# df = pd.read_csv(result, delimiter=',')
 
# print(calculate_bleu(pred, target))
result = 'testing_results/testing_s.csv'
data_root_path = '../dataset'

averages = evaluate(data_root_path, filename=result)

print(averages)

{'bleu_1': 0.4997301602867167, 'bleu_2': 0.34999124132530945, 'bleu_3': 0.23596186252701284, 'bleu_4': 0.167767887472968}


In [7]:
def calculate_sacrebleu_sentence(predicted, references):
    # Tokenizing the sentences
    predicted_tokens = tokenizer_indonesian(predicted)
    reference_tokens = [tokenizer_indonesian(ref) for ref in references]

    # Converting token lists back to strings for sacrebleu
    predicted_text = ' '.join(predicted_tokens)
    references_texts = [' '.join(ref) for ref in reference_tokens]

    # Calculate sentence BLEU score
    bleu_score = sacrebleu.sentence_bleu(predicted_text, references_texts)
    
    score = {
        'bleu_score' : bleu_score.score,
        'precisions' : bleu_score.precisions
    }
    return score

def evaluate_sacrebleu(data_root_path, filename):
    df = pd.read_csv(filename, delimiter=',')
    df['target'] = df['target'].apply(ast.literal_eval)
    
    scores = []
    for predicted, target in zip(df['predicted'], df['target']):
        score = calculate_sacrebleu_sentence(predicted, target)
        score = score['precisions']
        
        scores.append(score)
            
    # Calculate average BLEU score
    averages = {
        'bleu1': 0, 
        'bleu2': 0, 
        'bleu3': 0, 
        'bleu4': 0,
    }
    for i, _ in enumerate(scores):
        averages['bleu1'] += scores[i][0]
        averages['bleu2'] += scores[i][1]
        averages['bleu3'] += scores[i][2]
        averages['bleu4'] += scores[i][3]
     
    n = len(scores)
    averages['bleu1'] /= n
    averages['bleu2'] /= n
    averages['bleu3'] /= n
    averages['bleu4'] /= n   

    return {"average_bleu": averages}

# Usage example
result = 'testing_results/testing_m.csv'
data_root_path = '../dataset'

averages = evaluate_sacrebleu(data_root_path, result)
print(averages)

{'average_bleu': {'bleu1': 60.280876608771315, 'bleu2': 35.47049783549782, 'bleu3': 22.745539321789323, 'bleu4': 15.723918650793646}}


In [13]:
predicted_sentence = "gadis muda berpose di depan kamera ."
reference_sentences = ['Tiga anak bermain pasir di dekat pantai.', 'Tiga anak bermain di pasir di pantai', 'Tiga gadis bermain di pasir.', '""Tiga gadis kecil berambut pirang, dua berbaju biru, satu berbaju merah muda, bermain di pantai yang cerah.""', 'Tiga gadis muda menari di pantai di pasir.']

results = calculate_sacrebleu(predicted_sentence, reference_sentences)
print(results)

{'score': 15.619699684601283, 'precisions': [57.142857142857146, 16.666666666666668, 10.0, 6.25], 'bp': 1.0, 'sys_len': 7, 'ref_len': 7}


In [21]:
val = [
10.16,
14.44,
12.11,
11.51,
10.87,
10.91,
]
avg = 0.0
for v in val:
    avg += v
    
avg/= len(val)

avg

11.666666666666666